In [2]:
import warnings, os
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from scipy import stats
from scipy.stats.mstats import winsorize
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import QuantileTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
pd.set_option("display.max_columns", 60)

# ── 1. Adim: Veri Yukleme & Tekrar Eden Kayit Kontrolu ──────────────────────
DATA_PATH = "clinvar_conflicting.csv"
df = pd.read_csv(DATA_PATH, low_memory=False)

n_before = len(df)
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
n_after = len(df)

print(f"Toplam satir : {n_before}")
print(f"Tekil satir  : {n_after}")
print(f"Silinen tekrar: {n_before - n_after}")
print(f"Kolon sayisi : {df.shape[1]}")
print(f"\nHedef (CLASS) dagilimi:\n{df['CLASS'].value_counts()}")


Toplam satır: 65188
Tekilleştirilmiş satır: 65188
Silinen tekrar sayısı: 0
Yazıldı: /Users/ibrahim/Downloads/clinvar_conflicting_dedup.csv


In [ ]:
# ── 2. Adim: Eksik Deger & Uc Deger Haritasi ────────────────────────────────

# --- 2a. Eksik deger bar chart ---
missing_pct = df.isnull().mean().sort_values(ascending=False) * 100
missing_pct = missing_pct[missing_pct > 0]

fig, ax = plt.subplots(figsize=(14, 6))
missing_pct.plot.bar(ax=ax, color=sns.color_palette("OrRd_r", len(missing_pct)))
ax.set_ylabel("Eksik Oran (%)")
ax.set_title("Kolonlara Gore Eksik Deger Orani")
ax.axhline(y=50, ls="--", color="red", alpha=0.6, label="50% esik")
ax.legend()
plt.tight_layout()
plt.show()

print("Eksik deger ozet tablosu:")
missing_summary = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_pct": (df.isnull().mean() * 100).round(2),
    "dtype": df.dtypes,
    "nunique": df.nunique()
})
display(missing_summary.sort_values("missing_pct", ascending=False))

# --- 2b. Sayisal kolonlar icin boxplot (outlier haritasi) ---
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols_plot = [c for c in num_cols if c != "CLASS"]

fig, axes = plt.subplots(3, 5, figsize=(22, 12))
axes = axes.flatten()
for i, col in enumerate(num_cols_plot):
    if i >= len(axes):
        break
    df[col].dropna().plot.box(ax=axes[i], vert=True)
    axes[i].set_title(col, fontsize=10)
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
fig.suptitle("Sayisal Kolonlar - Boxplot (Outlier Haritasi)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── 3. Adim: Dolayli Sizinti Taramasi ────────────────────────────────────────

# 3a. Benzersiz ID niteligi olan kolonlar (satir basi unique --> leakage riski)
print("=== Yuksek Kardinalite (potential ID / leakage) ===")
for col in df.columns:
    ratio = df[col].nunique() / len(df)
    if ratio > 0.95:
        print(f"  {col}: {df[col].nunique()} unique / {len(df)} satir  ({ratio:.2%})")

# 3b. Ayni turev profil - allele frequency kolonlari
af_cols = ["AF_ESP", "AF_EXAC", "AF_TGP"]
print("\n=== Allele Frequency Korelasyonlari (ayni turev profil) ===")
print(df[af_cols].corr().round(4).to_string())

# 3c. CADD turevleri
cadd_cols = ["CADD_PHRED", "CADD_RAW"]
print(f"\n=== CADD_PHRED <-> CADD_RAW korelasyon: {df[cadd_cols].corr().iloc[0,1]:.4f} ===")

# 3d. Anormal kolonlar (neredeyse tamamen bos)
print("\n=== Anormal / Neredeyse Bos Kolonlar (>%95 missing) ===")
high_missing = (df.isnull().mean() > 0.95)
anormal_cols = high_missing[high_missing].index.tolist()
for col in anormal_cols:
    pct = df[col].isnull().mean() * 100
    print(f"  {col}: %{pct:.1f} bos")

# 3e. Drop kararlari
drop_leakage = ["CLNHGVS"]  # satir basi benzersiz --> ID
drop_anormal = [c for c in anormal_cols]  # %95+ bos kolonlar
drop_cadd_dup = ["CADD_RAW"]  # CADD_PHRED ile 0.96 korelasyon

cols_to_drop_stage1 = list(set(drop_leakage + drop_anormal + drop_cadd_dup))
print(f"\n=== Drop edilecek kolonlar ({len(cols_to_drop_stage1)} adet) ===")
for c in sorted(cols_to_drop_stage1):
    print(f"  - {c}")

df.drop(columns=cols_to_drop_stage1, inplace=True)
print(f"\nKalan kolon sayisi: {df.shape[1]}")

In [ ]:
# ── 4. Adim: Kolon Aileleri Kesfif (EDA) ─────────────────────────────────────

FEATURE_FAMILIES = {
    "genomic_position": [c for c in ["CHROM", "POS", "REF", "ALT"] if c in df.columns],
    "allele_frequency": [c for c in ["AF_ESP", "AF_EXAC", "AF_TGP"] if c in df.columns],
    "clinical_annotation": [c for c in ["CLNDISDB", "CLNDN", "CLNVC", "CLNVI", "MC", "ORIGIN"] if c in df.columns],
    "vep_annotation": [c for c in ["Allele", "Consequence", "IMPACT", "SYMBOL",
                                     "Feature_type", "Feature", "BIOTYPE", "EXON",
                                     "INTRON", "cDNA_position", "CDS_position",
                                     "Protein_position", "Amino_acids", "Codons",
                                     "STRAND", "BAM_EDIT"] if c in df.columns],
    "pathogenicity_score": [c for c in ["SIFT", "PolyPhen", "LoFtool", "CADD_PHRED",
                                         "BLOSUM62", "IMPACT", "Consequence"] if c in df.columns],
    "gene_info": [c for c in ["SYMBOL", "Feature", "Feature_type", "BIOTYPE"] if c in df.columns],
}

for family, cols in FEATURE_FAMILIES.items():
    print(f"[{family}] ({len(cols)} kolon): {cols}")

# --- 4a. Kardinalite raporu ---
print("\n=== Kardinalite Raporu ===")
card_df = pd.DataFrame({
    "nunique": df.nunique(),
    "dtype": df.dtypes,
    "missing_pct": (df.isnull().mean() * 100).round(2)
}).sort_values("nunique", ascending=False)
display(card_df)

# --- 4b. Sayisal kolon dagilimlari ---
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols_eda = [c for c in num_cols if c != "CLASS"]

fig, axes = plt.subplots(3, 5, figsize=(22, 12))
axes = axes.flatten()
for i, col in enumerate(num_cols_eda):
    if i >= len(axes):
        break
    df[col].dropna().hist(bins=50, ax=axes[i], edgecolor="black", alpha=0.7)
    skew_val = df[col].dropna().skew()
    axes[i].set_title(f"{col}\nskew={skew_val:.2f}", fontsize=9)
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
fig.suptitle("Sayisal Kolon Dagilimlari", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# --- 4c. Kategorik kolon dagilimlari (dusuk kardinalite) ---
cat_cols_low = [c for c in df.select_dtypes(include=["object", "string"]).columns
                if df[c].nunique() <= 10]

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()
for i, col in enumerate(cat_cols_low):
    if i >= len(axes):
        break
    df[col].value_counts().head(10).plot.barh(ax=axes[i])
    axes[i].set_title(col, fontsize=10)
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
fig.suptitle("Kategorik Kolon Dagilimlari (dusuk kardinalite)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# --- 4d. Korelasyon heatmap (numerik) ---
corr_matrix = df[num_cols_eda].corr()
fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, ax=ax, linewidths=0.5)
ax.set_title("Sayisal Kolonlar - Korelasyon Matrisi")
plt.tight_layout()
plt.show()

# --- 4e. Eksiklik patern matrisi ---
fig, ax = plt.subplots(figsize=(18, 8))
msno.matrix(df, ax=ax, sparkline=False, fontsize=8)
ax.set_title("Eksiklik Patern Matrisi")
plt.tight_layout()
plt.show()

In [ ]:
# ── 5. Adim: Fold Kurallari (CV Stratejisi) ──────────────────────────────────

# Sinif dengesizligi ~3:1 --> RepeatedStratifiedKFold
X_temp = df.drop(columns=["CLASS"])
y = df["CLASS"]

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

fold_stats = []
for fold_idx, (train_idx, val_idx) in enumerate(cv.split(X_temp, y)):
    y_train_fold = y.iloc[train_idx]
    y_val_fold = y.iloc[val_idx]
    fold_stats.append({
        "fold": fold_idx + 1,
        "train_size": len(train_idx),
        "val_size": len(val_idx),
        "train_class1_pct": (y_train_fold == 1).mean() * 100,
        "val_class1_pct": (y_val_fold == 1).mean() * 100,
    })

fold_df = pd.DataFrame(fold_stats)
print("=== RepeatedStratifiedKFold (5-fold x 3-repeat = 15 fold) ===")
print(f"Genel CLASS=1 orani: {(y == 1).mean() * 100:.2f}%\n")
display(fold_df.describe().round(2))

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(fold_df["fold"] - 0.2, fold_df["train_class1_pct"], width=0.4, label="Train CLASS=1 %")
ax.bar(fold_df["fold"] + 0.2, fold_df["val_class1_pct"], width=0.4, label="Val CLASS=1 %")
ax.axhline(y=(y == 1).mean() * 100, ls="--", color="red", label="Genel oran")
ax.set_xlabel("Fold")
ax.set_ylabel("CLASS=1 Orani (%)")
ax.set_title("Fold Bazinda Sinif Dagilimi (Stratified)")
ax.legend()
plt.tight_layout()
plt.show()

print("CV nesnesi hazir: cv (RepeatedStratifiedKFold, 5x3)")

In [ ]:
# ── 6. Adim: Eksik Deger Stratejisi ──────────────────────────────────────────

print("Imputation oncesi eksik deger sayisi:", df.isnull().sum().sum())

# 6a. Sayisal kolonlar --> median imputation
num_cols_impute = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols_impute = [c for c in num_cols_impute if c != "CLASS" and df[c].isnull().any()]

print(f"\nSayisal imputation ({len(num_cols_impute)} kolon, median):")
for col in num_cols_impute:
    median_val = df[col].median()
    n_miss = df[col].isnull().sum()
    df[col].fillna(median_val, inplace=True)
    print(f"  {col}: {n_miss} eksik -> median={median_val:.4f}")

# 6b. Kategorik kolonlar --> "missing" etiketi veya mod
# SIFT ve PolyPhen gibi %60+ bos kolonlarda "missing" bilgi tasiyabilir
cat_cols_impute = df.select_dtypes(include=["object", "string"]).columns.tolist()
cat_cols_impute = [c for c in cat_cols_impute if df[c].isnull().any()]

print(f"\nKategorik imputation ({len(cat_cols_impute)} kolon):")
for col in cat_cols_impute:
    n_miss = df[col].isnull().sum()
    miss_pct = n_miss / len(df) * 100
    if miss_pct > 30:
        df[col].fillna("missing", inplace=True)
        print(f"  {col}: {n_miss} eksik ({miss_pct:.1f}%) -> 'missing' etiketi")
    else:
        mode_val = df[col].mode()[0]
        df[col].fillna(mode_val, inplace=True)
        print(f"  {col}: {n_miss} eksik ({miss_pct:.1f}%) -> mod='{mode_val}'")

print(f"\nImputation sonrasi eksik deger sayisi: {df.isnull().sum().sum()}")
assert df.isnull().sum().sum() == 0, "Hala eksik deger var!"
print("Tum eksik degerler giderildi.")

In [ ]:
# ── 7. Adim: Aykiri Deger Bastirma (Winsorize) ──────────────────────────────

winsorize_cols = ["AF_ESP", "AF_EXAC", "AF_TGP", "CADD_PHRED",
                  "LoFtool", "ORIGIN", "POS",
                  "cDNA_position", "CDS_position", "Protein_position"]
winsorize_cols = [c for c in winsorize_cols if c in df.columns]

LOWER_LIMIT = 0.01
UPPER_LIMIT = 0.01

print("=== Winsorize (1. ve 99. persantil) ===\n")
fig, axes = plt.subplots(len(winsorize_cols), 2, figsize=(16, 4 * len(winsorize_cols)))

for i, col in enumerate(winsorize_cols):
    before = df[col].copy()
    lo = before.quantile(LOWER_LIMIT)
    hi = before.quantile(1 - UPPER_LIMIT)
    df[col] = df[col].clip(lower=lo, upper=hi)

    axes[i, 0].hist(before, bins=50, alpha=0.7, edgecolor="black")
    axes[i, 0].set_title(f"{col} - ONCE")
    axes[i, 0].axvline(lo, color="red", ls="--", alpha=0.7)
    axes[i, 0].axvline(hi, color="red", ls="--", alpha=0.7)

    axes[i, 1].hist(df[col], bins=50, alpha=0.7, edgecolor="black", color="green")
    axes[i, 1].set_title(f"{col} - SONRA")

    print(f"  {col}: clip [{lo:.4f}, {hi:.4f}]")

plt.suptitle("Winsorize Oncesi / Sonrasi", fontsize=14, y=1.005)
plt.tight_layout()
plt.show()

In [ ]:
# ── 8. Adim: Dagilim Donusumu ────────────────────────────────────────────────

# Carpik sayisal kolonlar icin log1p donusumu
skewed_cols = []
num_cols_now = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols_now = [c for c in num_cols_now if c != "CLASS"]

for col in num_cols_now:
    skew_val = df[col].skew()
    if abs(skew_val) > 1.0 and (df[col] >= 0).all():
        skewed_cols.append((col, skew_val))

print("=== Log1p Donusumu (|skew| > 1 ve degerler >= 0) ===\n")

if skewed_cols:
    n_plot = len(skewed_cols)
    fig, axes = plt.subplots(n_plot, 2, figsize=(14, 4 * n_plot))
    if n_plot == 1:
        axes = axes.reshape(1, -1)

    for i, (col, skew_before) in enumerate(skewed_cols):
        before = df[col].copy()
        df[col] = np.log1p(df[col])
        skew_after = df[col].skew()

        axes[i, 0].hist(before, bins=50, alpha=0.7, edgecolor="black")
        axes[i, 0].set_title(f"{col} ONCE (skew={skew_before:.2f})")

        axes[i, 1].hist(df[col], bins=50, alpha=0.7, edgecolor="black", color="orange")
        axes[i, 1].set_title(f"{col} SONRA log1p (skew={skew_after:.2f})")

        print(f"  {col}: skew {skew_before:.2f} -> {skew_after:.2f}")

    plt.suptitle("Log1p Donusumu Oncesi / Sonrasi", fontsize=14, y=1.005)
    plt.tight_layout()
    plt.show()
else:
    print("  Yuksek carpikliga sahip uygun kolon bulunamadi.")

# Kalan carpik kolonlar icin QuantileTransformer (rank-gauss)
still_skewed = []
for col in num_cols_now:
    if abs(df[col].skew()) > 1.0:
        still_skewed.append(col)

if still_skewed:
    print(f"\n=== QuantileTransformer (rank-gauss) - {len(still_skewed)} kolon ===")
    qt = QuantileTransformer(output_distribution="normal", random_state=42)
    df[still_skewed] = qt.fit_transform(df[still_skewed])
    for col in still_skewed:
        print(f"  {col}: rank-gauss uygulandi, yeni skew={df[col].skew():.2f}")
else:
    print("\nTum sayisal kolonlar makul carpiklik seviyesinde.")

In [ ]:
# ── 9. Adim: Dusuk Bilgi Kolon Elemesi ───────────────────────────────────────

print("=== Near-Zero Variance Filtresi ===\n")

# 9a. Sayisal kolonlarda VarianceThreshold
num_cols_check = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols_check = [c for c in num_cols_check if c != "CLASS"]

if num_cols_check:
    vt = VarianceThreshold(threshold=0.01)
    vt.fit(df[num_cols_check])
    low_var_mask = ~vt.get_support()
    low_var_num = [num_cols_check[i] for i in range(len(num_cols_check)) if low_var_mask[i]]
    if low_var_num:
        print(f"Sayisal near-zero variance kolonlar: {low_var_num}")
        df.drop(columns=low_var_num, inplace=True)
    else:
        print("Sayisal kolonlarda near-zero variance kolon yok.")

# 9b. Kategorik kolonlarda yuksek tekrar orani (>%99 tek deger)
cat_cols_check = df.select_dtypes(include=["object", "string"]).columns.tolist()
high_repeat_cols = []
for col in cat_cols_check:
    top_freq = df[col].value_counts(normalize=True).iloc[0]
    if top_freq > 0.99:
        high_repeat_cols.append((col, top_freq))
        print(f"  {col}: en sik deger orani = {top_freq:.4f} (>99%)")

if high_repeat_cols:
    drop_hr = [c for c, _ in high_repeat_cols]
    print(f"\nDrop edilen yuksek tekrar kolonlari: {drop_hr}")
    df.drop(columns=drop_hr, inplace=True)
else:
    print("Kategorik kolonlarda yuksek tekrar oranli kolon yok.")

print(f"\nKalan kolon sayisi: {df.shape[1]}")

In [ ]:
# ── 10. Adim: Korelasyon Kumeleri Cikarimi ────────────────────────────────────

num_cols_final = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols_final = [c for c in num_cols_final if c != "CLASS"]

corr = df[num_cols_final].corr().abs()

# Yuksek korelasyon ciftlerini bul (>0.80)
print("=== Yuksek Korelasyon Ciftleri (|r| > 0.80) ===\n")
high_corr_pairs = []
for i in range(len(corr.columns)):
    for j in range(i + 1, len(corr.columns)):
        if corr.iloc[i, j] > 0.80:
            pair = (corr.columns[i], corr.columns[j], corr.iloc[i, j])
            high_corr_pairs.append(pair)
            print(f"  {pair[0]} <-> {pair[1]}: {pair[2]:.4f}")

if not high_corr_pairs:
    print("  Yuksek korelasyonlu cift bulunamadi.")

# AF kolonlarini birlestir
af_remaining = [c for c in ["AF_ESP", "AF_EXAC", "AF_TGP"] if c in df.columns]
if len(af_remaining) > 1:
    print(f"\n=== AF Kolonlarini Birlestirme ===")
    df["AF_mean"] = df[af_remaining].mean(axis=1)
    df.drop(columns=af_remaining, inplace=True)
    print(f"  {af_remaining} -> tek kolon 'AF_mean' olarak birlestirildi.")

# CDS/cDNA/Protein position kontrolu
pos_cols = [c for c in ["cDNA_position", "CDS_position", "Protein_position"] if c in df.columns]
if len(pos_cols) > 1:
    pos_corr = df[pos_cols].corr()
    print(f"\n=== Position Korelasyonlari ===")
    print(pos_corr.round(4).to_string())
    high_pos_pairs = []
    for i in range(len(pos_cols)):
        for j in range(i + 1, len(pos_cols)):
            if abs(pos_corr.iloc[i, j]) > 0.80:
                high_pos_pairs.append((pos_cols[i], pos_cols[j]))
    if high_pos_pairs:
        keep_col = pos_cols[0]
        drop_pos = pos_cols[1:]
        df.drop(columns=drop_pos, inplace=True)
        print(f"  Yuksek korelasyon nedeniyle {drop_pos} drop edildi, {keep_col} tutuldu.")

# Guncel korelasyon heatmap
num_cols_updated = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols_updated = [c for c in num_cols_updated if c != "CLASS"]

if len(num_cols_updated) > 1:
    fig, ax = plt.subplots(figsize=(12, 8))
    corr_updated = df[num_cols_updated].corr()
    mask = np.triu(np.ones_like(corr_updated, dtype=bool))
    sns.heatmap(corr_updated, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
                center=0, ax=ax, linewidths=0.5)
    ax.set_title("Guncel Korelasyon Matrisi (temizlik sonrasi)")
    plt.tight_layout()
    plt.show()

print(f"\nFinal kolon sayisi: {df.shape[1]}")

In [ ]:
# ── 11. Adim: Ozellik Aile Etiketi Uretimi ───────────────────────────────────

FAMILY_DEFINITIONS = {
    "sequence":      ["REF", "ALT", "Allele", "Amino_acids", "Codons"],
    "position":      ["CHROM", "POS", "EXON", "INTRON", "cDNA_position",
                       "CDS_position", "Protein_position", "STRAND"],
    "frequency":     ["AF_ESP", "AF_EXAC", "AF_TGP", "AF_mean"],
    "clinical":      ["CLNDISDB", "CLNDN", "CLNVC", "CLNVI", "MC", "ORIGIN"],
    "pathogenicity": ["SIFT", "PolyPhen", "LoFtool", "CADD_PHRED", "BLOSUM62",
                       "IMPACT", "Consequence"],
    "gene":          ["SYMBOL", "Feature", "Feature_type", "BIOTYPE", "BAM_EDIT"],
}

current_cols = set(df.columns) - {"CLASS"}
feature_family_map = {}
for family, cols in FAMILY_DEFINITIONS.items():
    for col in cols:
        if col in current_cols:
            feature_family_map[col] = family

unassigned = current_cols - set(feature_family_map.keys())
for col in unassigned:
    feature_family_map[col] = "other"

# Ozet tablo
family_summary = pd.DataFrame([
    {"feature": col, "family": fam, "dtype": str(df[col].dtype)}
    for col, fam in sorted(feature_family_map.items(), key=lambda x: x[1])
])
display(family_summary)

# Aile bazinda kolon sayisi
family_counts = family_summary["family"].value_counts()
fig, ax = plt.subplots(figsize=(8, 5))
family_counts.plot.bar(ax=ax, edgecolor="black")
ax.set_title("Ozellik Ailelerine Gore Kolon Sayisi")
ax.set_ylabel("Kolon Sayisi")
plt.tight_layout()
plt.show()

# Final ozet
print(f"\n{'='*60}")
print(f"PIPELINE TAMAMLANDI")
print(f"{'='*60}")
print(f"Final veri seti boyutu : {df.shape[0]} satir x {df.shape[1]} kolon")
print(f"Hedef degisken (CLASS) : {df['CLASS'].value_counts().to_dict()}")
print(f"Eksik deger            : {df.isnull().sum().sum()}")
print(f"Ozellik aile sayisi    : {family_summary['family'].nunique()}")
print(f"CV stratejisi          : RepeatedStratifiedKFold(5x3)")
print(f"{'='*60}")